log版

In [3]:
import numpy as np
from scipy.sparse import coo_matrix, random
import json
from graphviz import Digraph

class Node:
    def __init__(self, op_type, children=None, scalar_pos=None, vector_row=None):
        self.op_type = op_type
        self.children = children or []
        self.scalar_pos = scalar_pos  # (i,j) 标量位置
        self.vector_row = vector_row  # 向量行号

    def to_dict(self):
        node_dict = {
            "type": self.op_type,
        }
        if self.op_type == 'row':
            node_dict["children"] = [c.to_dict() for c in self.children]
        elif self.op_type == 'add':
            node_dict["children"] = [c.to_dict() for c in self.children]
        elif self.op_type == 'mul':
            # 转换NumPy类型为Python原生类型
            node_dict["scalar_pos"] = (int(self.scalar_pos[0]), int(self.scalar_pos[1]))
            node_dict["vector_row"] = int(self.vector_row)
        return node_dict

def build_computation_graph(A_coo, B_dense):
    """使用COO格式的稀疏矩阵"""
    graph = {}
    M = A_coo.shape[0]
    
    # 按行分组非零元素
    from collections import defaultdict
    rows = defaultdict(list)
    for i, j, val in zip(map(int, A_coo.row),  # 转换为Python int
                        map(int, A_coo.col),
                        A_coo.data):
        rows[i].append((j, val))
    
    for i in range(M):
        row_node = Node('row')
        add_node = Node('add')
        row_node.children.append(add_node)
        
        if i in rows:
            for j, val in rows[i]:
                # 确保所有索引都是Python原生类型
                mul_node = Node('mul', 
                              scalar_pos=(int(i), int(j)), 
                              vector_row=int(j))
                add_node.children.append(mul_node)
        
        graph[f'C_{i}'] = row_node
    
    return graph

def visualize_graph(graph, filename='computation_graph'):
    dot = Digraph(comment='Computation Graph', format='png')
    dot.attr(rankdir='TB', splines='polyline')
    
    # 定义节点样式
    node_styles = {
        'row': {'shape': 'ellipse', 'color': 'lightblue', 'style': 'filled'},
        'add': {'shape': 'box', 'color': 'lightgreen', 'style': 'filled'},
        'mul': {'shape': 'diamond', 'color': 'orange', 'style': 'filled'}
    }
    
    # 遍历所有节点
    for c_row_key in graph:
        c_row = graph[c_row_key]
        row_num = c_row_key.split('_')[-1]
        
        # 行节点
        dot.node(f'row_{row_num}', 
                label=f'Row {row_num}', 
                **node_styles['row'])
        
        # 加法节点
        add_node = c_row.children[0]
        dot.node(f'add_{row_num}', 'Add', **node_styles['add'])
        dot.edge(f'row_{row_num}', f'add_{row_num}')
        
        # 乘法节点
        for idx, mul_node in enumerate(add_node.children):
            mul_id = f'mul_{row_num}_{idx}'
            label = f'Scalar: {mul_node.scalar_pos}\nVector: B[{mul_node.vector_row}]'
            dot.node(mul_id, label, **node_styles['mul'])
            dot.edge(f'add_{row_num}', mul_id)
    
    dot.render(filename, view=True)

def generate_test_case(M=16, N=16, density=0.1, seed=42):
    np.random.seed(seed)
    
    # 生成COO格式稀疏矩阵
    A_sparse = random(M, N, density=density, format='coo', random_state=seed)
    A_sparse.data = np.round(A_sparse.data * 10).astype(int)  # 转换为整数
    
    # 生成密集矩阵B
    B_dense = np.round(np.random.randn(N, N) * 2).astype(int)
    
    return A_sparse, B_dense

if __name__ == "__main__":
    A_coo, B_dense = generate_test_case(M=4, N=4, density=0.3, seed=42)
    
    # 构建计算图
    graph = build_computation_graph(A_coo, B_dense)
    
    # 打印图结构
    graph_dict = {k: v.to_dict() for k, v in graph.items()}
    print("图结构示例：")
    print(json.dumps(graph_dict, indent=2, ensure_ascii=False))
    
    # 可视化计算图
    visualize_graph(graph, filename='matrix_mult_graph')


图结构示例：
{
  "C_0": {
    "type": "row",
    "children": [
      {
        "type": "add",
        "children": [
          {
            "type": "mul",
            "scalar_pos": [
              0,
              0
            ],
            "vector_row": 0
          }
        ]
      }
    ]
  },
  "C_1": {
    "type": "row",
    "children": [
      {
        "type": "add",
        "children": [
          {
            "type": "mul",
            "scalar_pos": [
              1,
              0
            ],
            "vector_row": 0
          },
          {
            "type": "mul",
            "scalar_pos": [
              1,
              1
            ],
            "vector_row": 1
          },
          {
            "type": "mul",
            "scalar_pos": [
              1,
              3
            ],
            "vector_row": 3
          }
        ]
      }
    ]
  },
  "C_2": {
    "type": "row",
    "children": [
      {
        "type": "add",
        "children": [
        

Error: no "view" rule for type "image/png" passed its test case
       (for more information, add "--debug=1" on the command line)


合并树版

In [ ]:
import numpy as np
from scipy.sparse import coo_matrix, random
import json
from graphviz import Digraph
from collections import defaultdict

class Node:
    def __init__(self, op_type, children=None, scalar_pos=None, vector_row=None):
        self.op_type = op_type
        self.children = children or []
        self.scalar_pos = scalar_pos  # (i,j)
        self.vector_row = vector_row  # B的行号
        
    def to_dict(self):
        node_dict = {"type": self.op_type}
        if self.children:
            node_dict["children"] = [c.to_dict() for c in self.children]
        if self.scalar_pos:
            node_dict["scalar_pos"] = self.scalar_pos
        if self.vector_row is not None:
            node_dict["vector_row"] = self.vector_row
        return node_dict

def build_computation_graph(A_coo):
    """构建包含标量和向量子节点的计算图"""
    graph = {}
    M = A_coo.shape[0]
    
    # 按行分组非零元素
    rows = defaultdict(list)
    for i, j, val in zip(A_coo.row, A_coo.col, A_coo.data):
        rows[i].append((j, val))
    
    for i in range(M):
        row_node = Node('row')
        add_node = Node('add')
        row_node.children.append(add_node)
        
        if i in rows:
            for j, _ in rows[i]:
                # 创建乘法节点
                mul_node = Node('mul')
                
                # 创建标量子节点
                scalar_node = Node('scalar', scalar_pos=(i, j))
                
                # 创建向量子节点
                vector_node = Node('vector', vector_row=j)
                
                # 连接乘法节点的子节点
                mul_node.children = [scalar_node, vector_node]
                add_node.children.append(mul_node)
        
        graph[f'C_{i}'] = row_node
    
    return graph

def visualize_graph(graph, filename='computation_graph', merge_vectors=True):
    dot = Digraph(comment='Computation Graph', format='png')
    dot.attr(rankdir='TB', splines='polyline')
    
    # 节点样式配置
    styles = {
        'row': {'shape': 'ellipse', 'color': 'lightblue', 'style': 'filled'},
        'add': {'shape': 'box', 'color': 'lightgreen', 'style': 'filled'},
        'mul': {'shape': 'diamond', 'color': 'orange', 'style': 'filled'},
        'scalar': {'shape': 'note', 'color': 'yellow', 'style': 'filled'},
        'vector': {'shape': 'folder', 'color': 'pink', 'style': 'filled'}
    }
    
    # 用于跟踪已创建的向量节点
    vector_nodes = {}
    scalar_nodes = {}

    for row_key in graph:
        row_node = graph[row_key]
        row_num = row_key.split('_')[1]
        
        # 创建行节点
        dot.node(f'row_{row_num}', f'Row {row_num}', **styles['row'])
        
        # 处理加法节点
        if row_node.children:
            add_node = row_node.children[0]
            add_id = f'add_{row_num}'
            dot.node(add_id, 'Add', **styles['add'])
            dot.edge(f'row_{row_num}', add_id)
            
            # 处理乘法节点
            for mul_idx, mul_node in enumerate(add_node.children):
                mul_id = f'mul_{row_num}_{mul_idx}'
                dot.node(mul_id, 'Mul', **styles['mul'])
                dot.edge(add_id, mul_id)
                
                # 处理标量和向量子节点
                for child in mul_node.children:
                    if child.op_type == 'scalar':
                        # 标量节点唯一标识
                        scalar_id = f'scalar_{child.scalar_pos[0]}_{child.scalar_pos[1]}'
                        if scalar_id not in scalar_nodes:
                            dot.node(scalar_id, 
                                   f'A{child.scalar_pos}', 
                                   **styles['scalar'])
                            scalar_nodes[scalar_id] = True
                        dot.edge(mul_id, scalar_id)
                        
                    elif child.op_type == 'vector':
                        # 向量节点处理（可选合并）
                        vector_id = f'vector_{child.vector_row}'
                        if merge_vectors:
                            if vector_id not in vector_nodes:
                                dot.node(vector_id, 
                                       f'B[{child.vector_row}]', 
                                       **styles['vector'])
                                vector_nodes[vector_id] = True
                        else:
                            vector_id = f'vector_{row_num}_{mul_idx}'
                            dot.node(vector_id, 
                                   f'B[{child.vector_row}]', 
                                   **styles['vector'])
                        dot.edge(mul_id, vector_id)

    dot.render(filename, view=True, cleanup=True)

def generate_test_case(M=4, N=4, density=0.5, seed=42):
    np.random.seed(seed)
    A_sparse = random(M, N, density=density, format='coo', random_state=seed)
    A_sparse.data = np.round(A_sparse.data * 10).astype(int)
    return A_sparse

if __name__ == "__main__":
    # 生成测试数据（包含重复的向量行）
    A_coo = generate_test_case(M=4, N=4, density=0.6, seed=42)
    
    # 构建计算图
    graph = build_computation_graph(A_coo)
    
    # 可视化（启用向量节点合并）
    visualize_graph(graph, filename='merged_graph', merge_vectors=True)
    
    # 可视化（不启用合并）
    visualize_graph(graph, filename='separate_graph', merge_vectors=False)


Error: no "view" rule for type "image/png" passed its test case
       (for more information, add "--debug=1" on the command line)


Error: no "view" rule for type "image/png" passed its test case
       (for more information, add "--debug=1" on the command line)


In [15]:
import numpy as np
from scipy.sparse import coo_matrix, random
from graphviz import Digraph
from collections import defaultdict

class Node:
    def __init__(self, op_type, children=None, scalar_pos=None, scalar_val=None, vector_row=None):
        self.op_type = op_type
        self.children = children or []
        self.scalar_pos = scalar_pos  # (i,j)
        self.scalar_val = scalar_val  # 标量值
        self.vector_row = vector_row  # 向量行号
        
    def to_dict(self):
        node_dict = {"type": self.op_type}
        if self.children:
            node_dict["children"] = [c.to_dict() for c in self.children]
        if self.scalar_pos:
            node_dict["scalar_pos"] = self.scalar_pos
        if self.scalar_val is not None:
            node_dict["scalar_val"] = float(self.scalar_val)
        if self.vector_row is not None:
            node_dict["vector_row"] = int(self.vector_row)
        return node_dict

def build_computation_graph(A_coo, B_dense):
    """构建包含完整计算信息的图"""
    graph = {}
    M = A_coo.shape[0]
    rows = defaultdict(list)
    
    for i, j, val in zip(A_coo.row, A_coo.col, A_coo.data):
        rows[i].append((j, val))
    
    for i in range(M):
        row_node = Node('row')
        add_node = Node('add')
        row_node.children.append(add_node)
        
        if i in rows:
            for j, val in rows[i]:
                # 创建带实际值的标量节点
                scalar_node = Node('scalar', 
                                 scalar_pos=(i, j),
                                 scalar_val=val)
                # 创建带行号的向量节点
                vector_node = Node('vector',
                                 vector_row=j,
                                 scalar_val=tuple(B_dense[j].tolist()))  # 存储向量值
                # 创建乘法节点
                mul_node = Node('mul', children=[scalar_node, vector_node])
                add_node.children.append(mul_node)
        
        graph[f'C_{i}'] = row_node
    
    return graph

def compute_with_graph(graph, B_dense):
    """通过计算图执行计算"""
    M = len(graph)
    N = B_dense.shape[1]
    C = np.zeros((M, N))
    
    for row_key in graph:
        i = int(row_key.split('_')[1])
        add_node = graph[row_key].children[0]
        
        row_result = np.zeros(N)
        for mul_node in add_node.children:
            # 提取标量值和向量行
            scalar_val = mul_node.children[0].scalar_val
            vector = B_dense[mul_node.children[1].vector_row]
            row_result += scalar_val * vector
        
        C[i] = row_result
    
    return C

def visualize_graph(graph, B_dense, filename='computation_graph', merge_vectors=True):
    """图可视化"""
    # dot = Digraph(comment='Computation Graph', format='png')
    # dot.attr(rankdir='TB', splines='ortho')  
    
    dot = Digraph(engine='neato', comment='Computation Graph', format='png')
    dot.attr(mode="major", 
            overlap="scalexy",
            sep="+5")  # 节点间最小间隔
    
    # 样式配置
    styles = {
        'row': {'shape': 'ellipse', 'color': 'lightblue', 'style': 'filled'},
        'add': {'shape': 'box', 'color': 'lightgreen', 'style': 'filled'},
        'mul': {'shape': 'diamond', 'color': 'orange', 'style': 'filled', 'width': '1.2'},
        'scalar': {'shape': 'note', 'color': 'yellow', 'style': 'filled'},
        'vector': {'shape': 'cylinder', 'color': 'pink', 'style': 'filled'}
    }
    
    # 管理共享节点
    vector_nodes = {}
    scalar_nodes = {}
    
    # 先添加所有向量节点
    if merge_vectors:
        with dot.subgraph(name='cluster_vectors') as c:
            c.attr(label='B Matrix Rows', style='filled', color='lightgrey')
            for j in range(B_dense.shape[0]):
                vec_id = f'vector_{j}'
                c.node(vec_id, 
                    #   label=f'B[{j}]\n{tuple(B_dense[j].astype(int))}',
                      label=f'B[{j}]',
                      **styles['vector'])
                vector_nodes[j] = vec_id
    
    # 添加计算节点
    for row_key in sorted(graph.keys(), key=lambda x: int(x.split('_')[1])):
        row_node = graph[row_key]
        i = int(row_key.split('_')[1])
        
        # 行节点
        dot.node(f'row_{i}', f'C[{i}]', **styles['row'])
        
        # 加法节点
        add_id = f'add_{i}'
        dot.node(add_id, '+', **styles['add'])
        dot.edge(f'row_{i}', add_id)
        
        # 乘法节点
        for mul_idx, mul_node in enumerate(row_node.children[0].children):
            mul_id = f'mul_{i}_{mul_idx}'
            dot.node(mul_id, '×', **styles['mul'])
            dot.edge(add_id, mul_id)
            
            # 标量节点
            scalar_node = mul_node.children[0]
            scalar_id = f'scalar_{scalar_node.scalar_pos[0]}_{scalar_node.scalar_pos[1]}'
            if scalar_id not in scalar_nodes:
                dot.node(scalar_id,
                       f'A{scalar_node.scalar_pos}\n={scalar_node.scalar_val:.1f}',
                       **styles['scalar'])
                scalar_nodes[scalar_id] = True
            dot.edge(mul_id, scalar_id)
            
            # 向量节点
            vector_node = mul_node.children[1]
            if merge_vectors:
                vec_id = vector_nodes[vector_node.vector_row]
            else:
                vec_id = f'vector_{i}_{mul_idx}'
                dot.node(vec_id,
                       f'B[{vector_node.vector_row}]\n{tuple(vector_node.scalar_val)}',
                       **styles['vector'])
            dot.edge(mul_id, vec_id)
    
    dot.render(filename, view=True, cleanup=True)

def generate_test_case(M=4, N=4, density=0.3, seed=42):
    np.random.seed(seed)
    A_sparse = random(M, N, density=density, format='coo', random_state=seed)
    A_sparse.data = np.round(A_sparse.data * 10)
    B_dense = np.round(np.random.randn(N, N))
    return A_sparse, B_dense

if __name__ == "__main__":
    A_coo, B_dense = generate_test_case(M=16, N=16, density=0.1, seed=42)
    
    # 构建计算图
    graph = build_computation_graph(A_coo, B_dense)
    
    # 执行计算
    custom_C = compute_with_graph(graph, B_dense)
    
    # 验证结果
    scipy_C = A_coo.toarray() @ B_dense
    error = np.abs(custom_C - scipy_C).max()
    print(f"最大计算误差: {error:.2e}")
    print("结果验证:", "通过" if error < 1e-6 else "失败")
    
    # 可视化（合并模式）
    visualize_graph(graph, B_dense, filename='merged_view', merge_vectors=True)
    
    # 可视化（分离模式）
    visualize_graph(graph, B_dense, filename='separate_view', merge_vectors=False)


最大计算误差: 0.00e+00
结果验证: 通过


Error: no "view" rule for type "image/png" passed its test case
       (for more information, add "--debug=1" on the command line)
dot: graph is too large for cairo-renderer bitmaps. Scaling by 0.289722 to fit


Error: no "view" rule for type "image/png" passed its test case
       (for more information, add "--debug=1" on the command line)
